In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import scipy.optimize as opt
import qutip as qt
import os 
H = np.array([[1],[0]])
V = np.array([[0],[1]])
D = 1/np.sqrt(2)*(H+V)
A = 1/np.sqrt(2)*(H-V)
R = 1/np.sqrt(2)*(H+1j*V)
L = 1/np.sqrt(2)*(H-1j*V)

#mapping from basis labels to their corresponding state vectors
basismap = {
    'HH': np.kron(H, H),
    'HV': np.kron(H, V),
    'HD': np.kron(H, D),
    'HA': np.kron(H, A),
    'HR': np.kron(H, R),
    'HL': np.kron(H, L),
    'VH': np.kron(V, H),
    'VV': np.kron(V, V),
    'VD': np.kron(V, D),
    'VA': np.kron(V, A),
    'VR': np.kron(V, R),
    'VL': np.kron(V, L),
    'DH': np.kron(D, H),
    'DV': np.kron(D, V),
    'DD': np.kron(D, D),
    'DA': np.kron(D, A),
    'DR': np.kron(D, R),
    'DL': np.kron(D, L),
    'AH': np.kron(A, H),
    'AV': np.kron(A, V),
    'AD': np.kron(A, D),
    'AA': np.kron(A, A),
    'AR': np.kron(A, R),
    'AL': np.kron(A, L),
    'RH': np.kron(R, H),
    'RV': np.kron(R, V),
    'RD': np.kron(R, D),
    'RA': np.kron(R, A),
    'RR': np.kron(R, R),
    'RL': np.kron(R, L),
    'LH': np.kron(L, H),
    'LV': np.kron(L, V),
    'LD': np.kron(L, D),
    'LA': np.kron(L, A),
    'LR': np.kron(L, R),
    'LL': np.kron(L, L)
}

I = np.array([[1,0],[0,1]])
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])
pauli = [I,X,Y,Z]



# Q1

In [3]:
def read_measurements(filename):
    '''
    Helper function to load a file of measurment with corresponding basis, will generate a list with appropriate basis vectors (which can be useful for overcomplete bases)
    '''
    #get path
    path = os.path.join(os.getcwd(),filename)
    #load file
    df = pd.read_csv(path, delimiter = '\\s+', names = ["state","measurements"])
    #map basis letter/name to vector
    states = np.array(df['state'][1::])
    states = [basismap[b] for b in states]
    #make sure measurements are numerical
    measurements = np.array(df['measurements'][1::].astype('float'))
    return (states, measurements)

def rho_cholesky_2q(t):
    '''
    returns rho from cholesky decomposition for two qubits, t is a list of 16 parameters
    '''
    #Parametrized cholesky matrix
    T = np.array([[t[0],0,0,0],[t[1]+1j*t[2],t[3],0,0],[t[4]+1j*t[5],t[6]+1j*t[7],t[8],0],[t[9]+1j*t[10],t[11]+1j*t[12],t[13]+1j*t[14],t[15]]])
    #constructed valid density matrix (positive and trace 1)
    return (T.conj().T @ T)/(np.trace(T.conj().T @ T))

def count_objective(t, counts, projectors):
    '''
    loss function for photon count data (assumes poisonian error model)
    t: optimization parameters, list
    counts: measurements
    basis: list of basis states, should match 
    N: photon count rate
    projectors: list of projectors for measurements
    '''
    #initialize rho
    rho = rho_cholesky_2q(t)
    #initialize loss to 0    
    loss = 0
    for i in range(len(counts)):
        #expected counts according to born's
        predicted = t[16]*np.real(np.trace(projectors[i] @ rho))
        
        #ith term contributing to loss
        loss += (predicted-counts[i])**2/(2*predicted)
    return loss


def MLE_tomo(f, states, args):
    '''
    function to do MLE state tomography and reconstruct the appropriate density matrix

    f: loss/objective function
    t_init: inital parameter guess
    basis: list of measured states to construct projectors
    args: tuple with appropriate kwargs needed for loss/objective function
    '''
    #compute projectors (builds ordered list of projectors based on performed measurements)
    projectors = [np.outer(s,s.conj()) for s in states]
    #convert args to a tuple, add projectors to args
    args.append(projectors)
    args = tuple(args)

    
    # Set finite bounds for all parameters
    bounds = [(-1, 1) for _ in range(16)] + [(0, 1e7)]  # Initial guess for optimization parameters
    result = opt.dual_annealing(f, bounds=bounds, maxiter=10000, args=args)
    params = result.x
        #compute rho based on fit parameters
    rho = rho_cholesky_2q(params[0:16])
    print(f'Loss: {result.fun} \n')
    N = params[-1]
    with np.printoptions(3, suppress=False):
        if len(params) == 17:
            print(f'N Photons: {N} \n')
        print(f'MLE Reconstructed Density Matrix RHO: \n {rho} \n')
    return rho


#read measurements and reconstruct state (file has the data contained in the pre-lab)
states, counts = read_measurements('bell_measurements.txt')
t_init = np.random.rand(17)
rho = MLE_tomo(count_objective, states, [counts])

def bell_fidelity(rho):
    '''
    Function to compute fidelity of reconstructed state with bell state
    '''
    #define bell states in the computational basis
    bell_state00 = 1/np.sqrt(2)*(basismap['HH']+basismap['VV'])
    bell_state01 = 1/np.sqrt(2)*(basismap['HH']-basismap['VV'])
    bell_state10 = 1/np.sqrt(2)*(basismap['HV']+basismap['VH'])
    bell_state11 = 1/np.sqrt(2)*(basismap['HV']-basismap['VH'])

    print(f'Fidelity with Bell State 00: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state00))}')
    print(f'Fidelity with Bell State 01: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state01))}')
    print(f'Fidelity with Bell State 10: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state10))}')
    print(f'Fidelity with Bell State 11: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state11))}')

bell_fidelity(rho)


Loss: 343.9059364212003 

N Photons: 71510.01746807198 

MLE Reconstructed Density Matrix RHO: 
 [[ 0.503+0.j    -0.021-0.011j -0.025+0.019j  0.466-0.022j]
 [-0.021+0.011j  0.005+0.j     0.004+0.002j -0.033+0.006j]
 [-0.025-0.019j  0.004-0.002j  0.007+0.j    -0.04 -0.011j]
 [ 0.466+0.022j -0.033-0.006j -0.04 +0.011j  0.484+0.j   ]] 

Fidelity with Bell State 00: 0.979772843894571
Fidelity with Bell State 01: 0.1660785428037268
Fidelity with Bell State 10: 0.10158006103530785
Fidelity with Bell State 11: 0.04630964927366122


# Q2

In [4]:
def bell_fidelity(rho):
    '''
    Function to compute fidelity of reconstructed state with bell state
    '''
    #define bell states in the computational basis
    bell_state00 = 1/np.sqrt(2)*(basismap['HH']+basismap['VV'])
    bell_state01 = 1/np.sqrt(2)*(basismap['HH']-basismap['VV'])
    bell_state10 = 1/np.sqrt(2)*(basismap['HV']+basismap['VH'])
    bell_state11 = 1/np.sqrt(2)*(basismap['HV']-basismap['VH'])

    print(f'Fidelity with Bell State 00: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state00))}')
    print(f'Fidelity with Bell State 01: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state01))}')
    print(f'Fidelity with Bell State 10: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state10))}')
    print(f'Fidelity with Bell State 11: {qt.fidelity(qt.Qobj(rho), qt.Qobj(bell_state11))}')

states, counts = read_measurements('bell_tomo_4thmarch.txt')
rho = MLE_tomo(count_objective, states, [counts])
bell_fidelity(rho)

Loss: 233.6456362657176 

N Photons: 244439.2395713008 

MLE Reconstructed Density Matrix RHO: 
 [[ 0.49 +0.j     0.001-0.006j -0.003-0.004j  0.479-0.074j]
 [ 0.001+0.006j  0.01 +0.j    -0.005+0.009j  0.012+0.006j]
 [-0.003+0.004j -0.005-0.009j  0.011+0.j    -0.007-0.005j]
 [ 0.479+0.074j  0.012-0.006j -0.007+0.005j  0.489+0.j   ]] 

Fidelity with Bell State 00: 0.983999330312499
Fidelity with Bell State 01: 0.1042104341063384
Fidelity with Bell State 10: 0.0742484302237478
Fidelity with Bell State 11: 0.12398658863463648


# Q3

In [5]:
def spherical_to_cartesian(theta, phi):
    '''
    helper function to convert spherical coordinates to cartesian coordinates (for bell correlation measurements)
    '''
    x = np.sin(theta)*np.cos(phi)
    y = np.sin(theta)*np.sin(phi)
    z = np.cos(theta)
    return np.array([x,y,z])

def bell_correlations(rho, n: np.ndarray, m: np.ndarray):
    '''
    helper function to compute bell correlations for given measurement settings n and m (should be 3d vectors)
    '''
    #compute correlation for each setting
    sigma_n = n[0]*X + n[1]*Y + n[2]*Z
    sigma_m = m[0]*X + m[1]*Y + m[2]*Z
    return np.real(np.trace((np.kron(sigma_n, sigma_m)) @ rho))



# Q4

In [8]:
def chsh_bell_parameter(rho, a, a_prime, b, b_prime):
    '''
    expected CHSH Bell parameter S for arbitrary measurement settings
    S = |E(a,b) + E(a,b') + E(a',b) - E(a',b')|
    '''
    e_ab = bell_correlations(rho, a, b)
    e_abp = bell_correlations(rho, a, b_prime)
    e_apb = bell_correlations(rho, a_prime, b)
    e_apbp = bell_correlations(rho, a_prime, b_prime)
    return np.abs(e_ab + e_abp + e_apb - e_apbp)

def optimize_chsh(rho):
    '''
    function to optimize CHSH Bell parameter S over measurement settings, returns optimal S and corresponding settings
    '''
    def objective(params):
        #objective function to minimize (negative of CHSH Bell parameter)
        a = spherical_to_cartesian(params[0], params[1])
        a_prime = spherical_to_cartesian(params[2], params[3])
        b = spherical_to_cartesian(params[4], params[5])
        b_prime = spherical_to_cartesian(params[6], params[7])
        return -chsh_bell_parameter(rho, a, a_prime, b, b_prime)

    # Use differential_evolution instead of minimize with Powell method
    bounds = [(0, np.pi) for _ in range(8)]
    result = opt.dual_annealing(objective, bounds=bounds)   
    optimal_params = result.x
    optimal_S = -result.fun

    #convert optimal parameters back to cartesian coordinates for measurement settings
    a_opt = spherical_to_cartesian(optimal_params[0], optimal_params[1])
    a_prime_opt = spherical_to_cartesian(optimal_params[2], optimal_params[3])
    b_opt = spherical_to_cartesian(optimal_params[4], optimal_params[5])
    b_prime_opt = spherical_to_cartesian(optimal_params[6], optimal_params[7])

    #compute optimal measurement operators for optimal settings (didn't end up using these, just printing them for fun)
    sigma_a_opt = a_opt[0]*X + a_opt[1]*Y + a_opt[2]*Z
    sigma_a_prime_opt = a_prime_opt[0]*X + a_prime_opt[1]*Y + a_prime_opt[2]*Z
    sigma_b_opt = b_opt[0]*X + b_opt[1]*Y + b_opt[2]*Z
    sigma_b_prime_opt = b_prime_opt[0]*X + b_prime_opt[1]*Y + b_prime_opt[2]*Z

    with np.printoptions(precision=3, suppress=True):
        print(f'Optimal CHSH Bell Parameter S: {optimal_S}')
        print(f'a: {a_opt} \n \n sigma_a: {sigma_a_opt}\n')
        print(f'a_prime: {a_prime_opt} \n \n sigma_a_prime: {sigma_a_prime_opt} \n')
        print(f'b: {b_opt} \n \n sigma_b: {sigma_b_opt} \n')
        print(f'b_prime: {b_prime_opt} \n \n sigma_b_prime: {sigma_b_prime_opt} \n')
    return (a_opt, a_prime_opt, b_opt, b_prime_opt)
       
bell00 = 1/np.sqrt(2)*(basismap['HH']+basismap['VV'])
HH = np.kron(H, H)

In [20]:
optimize_chsh(np.kron(bell00, bell00.conj().T))


Optimal CHSH Bell Parameter S: 2.828427124745601
a: [ 0.067  0.766 -0.639] 
 
 sigma_a: [[-0.639+0.j     0.067-0.766j]
 [ 0.067+0.766j  0.639+0.j   ]]

a_prime: [-0.039  0.642  0.765] 
 
 sigma_a_prime: [[ 0.765+0.j    -0.039-0.642j]
 [-0.039+0.642j -0.765+0.j   ]] 

b: [-0.02   0.996 -0.089] 
 
 sigma_b: [[-0.089+0.j    -0.02 -0.996j]
 [-0.02 +0.996j  0.089+0.j   ]] 

b_prime: [-0.075  0.087  0.993] 
 
 sigma_b_prime: [[ 0.993+0.j    -0.075-0.087j]
 [-0.075+0.087j -0.993+0.j   ]] 



(array([ 0.06699252,  0.76598031, -0.63936387]),
 array([-0.03908496,  0.64232642,  0.76543395]),
 array([-0.01973375,  0.99582312, -0.08914529]),
 array([-0.07500782,  0.08743573,  0.99334225]))

In [21]:
optimize_chsh(np.kron(HH, HH.conj().T))

Optimal CHSH Bell Parameter S: 1.9999999999999651
a: [ 0.  0. -1.] 
 
 sigma_a: [[-1.+0.j  0.-0.j]
 [ 0.+0.j  1.+0.j]]

a_prime: [ 0.177  0.171 -0.969] 
 
 sigma_a_prime: [[-0.969+0.j     0.177-0.171j]
 [ 0.177+0.171j  0.969+0.j   ]] 

b: [ 0.  0. -1.] 
 
 sigma_b: [[-1.+0.j  0.-0.j]
 [ 0.+0.j  1.+0.j]] 

b_prime: [ 0.  0. -1.] 
 
 sigma_b_prime: [[-1.+0.j  0.-0.j]
 [ 0.+0.j  1.+0.j]] 



(array([ 6.75287400e-17,  1.02163923e-16, -1.00000000e+00]),
 array([ 0.1769676 ,  0.17056433, -0.96932465]),
 array([ 1.14830453e-16,  4.25624809e-17, -1.00000000e+00]),
 array([ 1.52594263e-07,  1.50515715e-06, -1.00000000e+00]))

In [22]:
optimize_chsh(rho)

Optimal CHSH Bell Parameter S: 2.7181146172086055
a: [ 0.85   0.435 -0.297] 
 
 sigma_a: [[-0.297+0.j     0.85 -0.435j]
 [ 0.85 +0.435j  0.297+0.j   ]]

a_prime: [-0.422  0.208 -0.883] 
 
 sigma_a_prime: [[-0.883+0.j    -0.422-0.208j]
 [-0.422+0.208j  0.883+0.j   ]] 

b: [-0.285  0.425  0.859] 
 
 sigma_b: [[ 0.859+0.j    -0.285-0.425j]
 [-0.285+0.425j -0.859+0.j   ]] 

b_prime: [-0.897  0.11  -0.427] 
 
 sigma_b_prime: [[-0.427+0.j   -0.897-0.11j]
 [-0.897+0.11j  0.427+0.j  ]] 



(array([ 0.84993802,  0.43500406, -0.2972824 ]),
 array([-0.42170683,  0.20770424, -0.8826224 ]),
 array([-0.28527983,  0.42474889,  0.85918787]),
 array([-0.89747588,  0.10972025, -0.42719845]))

# Q5

In [7]:
def get_bench_params(n):
    '''function to compute optimal HWP and QWP angles for a given measurement setting n (should be a 3d vector)
    '''
    def to_spherical(cartesian):
        '''helper funcion to convert cartesian coordinates to spherical coordinates'''
        x = cartesian[0]
        y = cartesian[1]
        z = cartesian[2]
        theta = np.arccos(z)
        phi = np.arctan2(y, x)
        return (theta, phi)
    theta, phi = to_spherical(n)

    zero = qt.Qobj([[1],[0]])
    one = qt.Qobj([[0],[1]])
    measure_state = np.cos(theta/2)*zero + np.sin(theta/2)*np.exp(1j*phi)*one

    
    def objective(params):
        #objective function to minimize (negative of fidelity with target measurement state) of HWP and QWP transformations on the input state and H. 
        HWP = qt.Qobj([[np.cos(2*params[0]), np.sin(2*params[0])],[np.sin(2*params[0]), -np.cos(2*params[0])]])
        QWP = 1/np.sqrt(2)*qt.Qobj([[1+1j*np.cos(2*params[1]), 1j*np.sin(2*params[1])],[1j*np.sin(2*params[1]), 1-1j*np.cos(2*params[1])]])
        return -qt.fidelity(HWP @ QWP @ measure_state, zero)
    
    
    #bound param[0] to [0, pi/2) and param[1] to [0, pi)
    bounds = [(0, np.pi/2), (0, np.pi)]
    result = opt.dual_annealing(objective, bounds=bounds)
    optimal_params = result.x
    theta1, theta2 = optimal_params
    print(f'Optimal HWP angle: {theta1*180/np.pi} degrees')
    print(f'Optimal QWP angle: {theta2*180/np.pi} degrees')
    print(f'Optimal Fidelity: {-result.fun} \n')
    return optimal_params

#get_bench_params(np.array([1,0,0]))

unit_vecs = optimize_chsh(rho)

for i in unit_vecs:
    get_bench_params(i)

Optimal CHSH Bell Parameter S: 2.755599025429425
a: [-0.579  0.815  0.01 ] 
 
 sigma_a: [[ 0.01 +0.j    -0.579-0.815j]
 [-0.579+0.815j -0.01 +0.j   ]]

a_prime: [-0.008  0.012  1.   ] 
 
 sigma_a_prime: [[ 1.   +0.j    -0.008-0.012j]
 [-0.008+0.012j -1.   +0.j   ]] 

b: [ 0.308  0.634 -0.71 ] 
 
 sigma_b: [[-0.71 +0.j     0.308-0.634j]
 [ 0.308+0.634j  0.71 +0.j   ]] 

b_prime: [0.337 0.65  0.681] 
 
 sigma_b_prime: [[ 0.681+0.j    0.337-0.65j]
 [ 0.337+0.65j -0.681+0.j  ]] 

Optimal HWP angle: 81.3916797614542 degrees
Optimal QWP angle: 135.4906550936003 degrees
Optimal Fidelity: 0.9999999999880117 

Optimal HWP angle: 89.71019924648103 degrees
Optimal QWP angle: 89.7652473995759 degrees
Optimal Fidelity: 0.9999999999999779 

Optimal HWP angle: 29.303203346509708 degrees
Optimal QWP angle: 168.2718452988036 degrees
Optimal Fidelity: 0.9999999999997908 

Optimal HWP angle: 16.71169085596551 degrees
Optimal QWP angle: 13.146836491219497 degrees
Optimal Fidelity: 0.9999999999888304 



In [39]:
#check that I can get the bell parameter for my reconstructed state from the optimal settings
unit_vecs = optimize_chsh(rho)

def unitary(theta1, theta2):
    HWP = qt.Qobj([[np.cos(2*theta1), np.sin(2*theta1)],[np.sin(2*theta1), -np.cos(2*theta1)]])
    QWP = 1/np.sqrt(2)*qt.Qobj([[1+1j*np.cos(2*theta2), 1j*np.sin(2*theta2)],[1j*np.sin(2*theta2), 1-1j*np.cos(2*theta2)]])
    return QWP @ HWP

sigmas = []
#reconstruct sigma_n for each optimal setting
for i in unit_vecs:
    optimal_params = get_bench_params(i)
    sigman = unitary(optimal_params[0], optimal_params[1]) @ qt.sigmaz() @ unitary(optimal_params[0], optimal_params[1]).dag()
    sigmas.append(sigman)

#compute bell parameter for reconstructed state using optimal settings
e_ab = bell_correlations(rho, unit_vecs[0], unit_vecs[2])
e_abp = bell_correlations(rho, unit_vecs[0], unit_vecs[3])
e_apb = bell_correlations(rho, unit_vecs[1], unit_vecs[2])
e_apbp = bell_correlations(rho, unit_vecs[1], unit_vecs[3])
S = np.abs(e_ab + e_abp + e_apb - e_apbp)
print(f'Bell Parameter S for Reconstructed State using Optimal Settings: {S}')

Optimal CHSH Bell Parameter S: 2.7079230726315386
a: [-0.229  0.578  0.783] 
 
 sigma_a: [[ 0.783+0.j    -0.229-0.578j]
 [-0.229+0.578j -0.783+0.j   ]]

a_prime: [ 0.624 -0.482  0.615] 
 
 sigma_a_prime: [[ 0.615+0.j     0.624+0.482j]
 [ 0.624-0.482j -0.615+0.j   ]] 

b: [-0.297  0.051 -0.953] 
 
 sigma_b: [[-0.953+0.j    -0.297-0.051j]
 [-0.297+0.051j  0.953+0.j   ]] 

b_prime: [ 0.592  0.8   -0.096] 
 
 sigma_b_prime: [[-0.096+0.j   0.592-0.8j]
 [ 0.592+0.8j  0.096+0.j ]] 

Optimal HWP angle: 99.11055191995939 degrees
Optimal QWP angle: 0.003677071410760272 degrees
Optimal Fidelity: 0.993306766423481 

Optimal HWP angle: 94.15292038140575 degrees
Optimal QWP angle: 22.700413565384846 degrees
Optimal Fidelity: 0.9999999999091581 

Optimal HWP angle: 48.595293677399376 degrees
Optimal QWP angle: 8.655304807192504 degrees
Optimal Fidelity: 0.9999999999999986 

Optimal HWP angle: 128.09963010850385 degrees
Optimal QWP angle: 49.6318172572536 degrees
Optimal Fidelity: 0.9999999907337634 


In [24]:
states, counts = read_measurements('state_tomo_bell.txt')
t_init = np.random.rand(17)
rho = MLE_tomo(count_objective, t_init, states, [counts])
bell_fidelity(rho)

Loss: 236.74699795411343 

N Photons: 240356.4007699962 

MLE Reconstructed Density Matrix RHO: 
 [[ 0.476+0.j     0.017+0.005j -0.001+0.012j  0.481-0.037j]
 [ 0.017-0.005j  0.012+0.j    -0.007+0.01j   0.025-0.012j]
 [-0.001-0.012j -0.007-0.01j   0.013+0.j    -0.012-0.016j]
 [ 0.481+0.037j  0.025+0.012j -0.012+0.016j  0.5  +0.j   ]] 

Fidelity with Bell State 00: 0.9846011961273924
Fidelity with Bell State 01: 0.08077910003165296
Fidelity with Bell State 10: 0.07405954863235169
Fidelity with Bell State 11: 0.1361998709212178


In [30]:
states, counts = read_measurements('bell_tomo_4thmarch.txt')
t_init = np.random.rand(17)
rho = MLE_tomo(count_objective, states, [counts])
bell_fidelity(rho)

unit_vecs = optimize_chsh(rho)

for i in unit_vecs:
    get_bench_params(i)
    

Loss: 233.645636830025 

N Photons: 244439.80096591113 

MLE Reconstructed Density Matrix RHO: 
 [[ 0.49 +0.j     0.001-0.006j -0.003-0.004j  0.479-0.074j]
 [ 0.001+0.006j  0.01 +0.j    -0.005+0.009j  0.012+0.006j]
 [-0.003+0.004j -0.005-0.009j  0.011+0.j    -0.007-0.005j]
 [ 0.479+0.074j  0.012-0.006j -0.007+0.005j  0.489+0.j   ]] 

Fidelity with Bell State 00: 0.9839994459125453
Fidelity with Bell State 01: 0.10420930139927927
Fidelity with Bell State 10: 0.07424836910107653
Fidelity with Bell State 11: 0.12398665982890693
Optimal CHSH Bell Parameter S: 2.7556003771414117
a: [-0.581  0.813  0.029] 
 
 sigma_a: [[ 0.029+0.j    -0.581-0.813j]
 [-0.581+0.813j -0.029+0.j   ]]

a_prime: [0.011 0.    1.   ] 
 
 sigma_a_prime: [[ 1.   +0.j  0.011-0.j]
 [ 0.011+0.j -1.   +0.j]] 

b: [ 0.298  0.623 -0.723] 
 
 sigma_b: [[-0.723+0.j     0.298-0.623j]
 [ 0.298+0.623j  0.723+0.j   ]] 

b_prime: [0.349 0.659 0.666] 
 
 sigma_b_prime: [[ 0.666+0.j     0.349-0.659j]
 [ 0.349+0.659j -0.666+0.j   ]] 

Matt measurements

In [32]:
correlation_ab = np.array([[16319, 112872], [105225, 16000]])
correlation_abp = np.array([[22071, 102976], [98276, 23152]])
correlation_apb = np.array([[30296, 97810], [90549, 30145]])
correlation_apbp = np.array([[103792, 19261], [18514, 103974]])


def calculate_chsh_from_counts(correlation_ab, correlation_abp, correlation_apb, correlation_apbp):
    e_ab = (correlation_ab[0,0] + correlation_ab[1,1] - correlation_ab[0,1] - correlation_ab[1,0])/(correlation_ab.sum())
    e_abp = (correlation_abp[0,0] + correlation_abp[1,1] - correlation_abp[0,1] - correlation_abp[1,0])/(correlation_abp.sum())
    e_apb = (correlation_apb[0,0] + correlation_apb[1,1] - correlation_apb[0,1] - correlation_apb[1,0])/(correlation_apb.sum())
    e_apbp = (correlation_apbp[0,0] + correlation_apbp[1,1] - correlation_apbp[0,1] - correlation_apbp[1,0])/(correlation_apbp.sum())
    S = np.abs(e_ab + e_abp + e_apb - e_apbp)
    print(f'Bell Parameter S from Counts: {S}')

calculate_chsh_from_counts(correlation_ab, correlation_abp, correlation_apb, correlation_apbp)



Bell Parameter S from Counts: 2.581371356081866


Jer measurements

In [33]:
correlation_ab = np.array([[14425, 105469], [108541, 14000]])
correlation_abp = np.array([[30569, 90854], [94993, 29468]])
correlation_apb = np.array([[23019, 99443], [101684, 21843]])
correlation_apbp = np.array([[110660, 15348], [16865, 106531]])

calculate_chsh_from_counts(correlation_ab, correlation_abp, correlation_apb, correlation_apbp)

Bell Parameter S from Counts: 2.654100340780148
